# 🛒 E-Commerce Intelligence Platform
## Phase 2 — Data Cleaning & Preprocessing
**Goal:** Fix missing values, data types, duplicates & engineer new features  
**Analyst:** Shivansh Pandey

In [1]:
# ============================================================
# CELL 1 — IMPORTS & LOAD RAW DATA
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Upload files (run this and upload all 9 CSVs again)
from google.colab import files
uploaded = files.upload()

print("✅ Libraries imported & files uploaded")

Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv
✅ Libraries imported & files uploaded


In [2]:
# ============================================================
# CELL 2 — LOAD ALL DATASETS
# ============================================================

orders       = pd.read_csv('olist_orders_dataset.csv')
order_items  = pd.read_csv('olist_order_items_dataset.csv')
payments     = pd.read_csv('olist_order_payments_dataset.csv')
reviews      = pd.read_csv('olist_order_reviews_dataset.csv')
customers    = pd.read_csv('olist_customers_dataset.csv')
products     = pd.read_csv('olist_products_dataset.csv')
sellers      = pd.read_csv('olist_sellers_dataset.csv')
geolocation  = pd.read_csv('olist_geolocation_dataset.csv')
category     = pd.read_csv('product_category_name_translation.csv')

print("✅ All datasets loaded")

✅ All datasets loaded


In [3]:
# ============================================================
# CELL 3 — DETAILED MISSING VALUES REPORT
# ============================================================

def missing_report(df, name):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"✅ {name} — No missing values\n")
        return
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    print(f"\n⚠️  {name.upper()} — Missing Values:")
    print(report)
    print()

datasets = {
    'orders'      : orders,
    'order_items' : order_items,
    'payments'    : payments,
    'reviews'     : reviews,
    'customers'   : customers,
    'products'    : products,
    'sellers'     : sellers,
    'geolocation' : geolocation,
    'category'    : category
}

for name, df in datasets.items():
    missing_report(df, name)


⚠️  ORDERS — Missing Values:
                               Missing Count  Missing %
order_approved_at                        160       0.16
order_delivered_carrier_date            1783       1.79
order_delivered_customer_date           2965       2.98

✅ order_items — No missing values

✅ payments — No missing values


⚠️  REVIEWS — Missing Values:
                        Missing Count  Missing %
review_comment_title            87656      88.34
review_comment_message          58247      58.70

✅ customers — No missing values


⚠️  PRODUCTS — Missing Values:
                            Missing Count  Missing %
product_category_name                 610       1.85
product_name_lenght                   610       1.85
product_description_lenght            610       1.85
product_photos_qty                    610       1.85
product_weight_g                        2       0.01
product_length_cm                       2       0.01
product_height_cm                       2       0.01
product_wi

In [4]:
# ============================================================
# CELL 4 — CLEAN ORDERS TABLE
# ============================================================

print(f"Orders shape before cleaning: {orders.shape}")

# Step 1 — Convert all date columns to datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Step 2 — Fill missing approval times with purchase time
# Logic: if order was never approved, use purchase time as proxy
orders['order_approved_at'] = orders['order_approved_at'].fillna(
    orders['order_purchase_timestamp']
)

# Step 3 — Keep only delivered orders for time-based analysis
# (cancelled/unavailable orders skew delivery metrics)
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# Step 4 — Remove rows where delivery date is missing (can't analyze)
orders_delivered = orders_delivered.dropna(
    subset=['order_delivered_customer_date']
)

# Step 5 — Engineer new features
# Delivery time in days
orders_delivered['delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

# Was the order delivered on time?
orders_delivered['is_late'] = (
    orders_delivered['order_delivered_customer_date'] >
    orders_delivered['order_estimated_delivery_date']
).astype(int)

# Days early or late (negative = early, positive = late)
orders_delivered['days_diff'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_estimated_delivery_date']
).dt.days

print(f"Orders shape after cleaning : {orders_delivered.shape}")
print(f"\n📊 Delivery Stats:")
print(f"  Average delivery time : {orders_delivered['delivery_days'].mean():.1f} days")
print(f"  Late deliveries       : {orders_delivered['is_late'].sum():,} ({orders_delivered['is_late'].mean()*100:.1f}%)")
print(f"  On-time deliveries    : {(orders_delivered['is_late']==0).sum():,} ({(1-orders_delivered['is_late'].mean())*100:.1f}%)")

Orders shape before cleaning: (99441, 8)
Orders shape after cleaning : (96470, 11)

📊 Delivery Stats:
  Average delivery time : 12.1 days
  Late deliveries       : 7,826 (8.1%)
  On-time deliveries    : 88,644 (91.9%)


In [5]:
# ============================================================
# CELL 5 — CLEAN PRODUCTS TABLE
# ============================================================

print(f"Products shape before: {products.shape}")

# Step 1 — Merge with English category names
products = products.merge(category, on='product_category_name', how='left')

# Step 2 — Fill missing English category names
products['product_category_name_english'] = (
    products['product_category_name_english']
    .fillna('unknown')
)

# Step 3 — Fill missing product dimensions with median
# (median is better than mean for skewed data)
dimension_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in dimension_cols:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)

# Step 4 — Fill missing name length and description length
products['product_name_lenght'] = products['product_name_lenght'].fillna(
    products['product_name_lenght'].median()
)
products['product_description_lenght'] = products['product_description_lenght'].fillna(
    products['product_description_lenght'].median()
)

print(f"Products shape after : {products.shape}")
print(f"✅ Products cleaned successfully")

Products shape before: (32951, 9)
Products shape after : (32951, 10)
✅ Products cleaned successfully


In [6]:
# ============================================================
# CELL 6 — CLEAN REVIEWS TABLE
# ============================================================

print(f"Reviews shape before: {reviews.shape}")

# Step 1 — Fill missing review comments with empty string
reviews['review_comment_title']   = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

# Step 2 — Convert dates
reviews['review_creation_date']        = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp']     = pd.to_datetime(reviews['review_answer_timestamp'])

# Step 3 — Remove duplicate reviews (keep latest per order)
reviews = reviews.sort_values('review_answer_timestamp', ascending=False)
reviews = reviews.drop_duplicates(subset='order_id', keep='first')

print(f"Reviews shape after : {reviews.shape}")
print(f"\n📊 Review Score Distribution:")
print(reviews['review_score'].value_counts().sort_index())

Reviews shape before: (99224, 7)
Reviews shape after : (98673, 7)

📊 Review Score Distribution:
review_score
1    11363
2     3131
3     8133
4    19038
5    57008
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 7 — BUILD MASTER DATASET
# Merge all tables into one analysis-ready dataframe
# ============================================================

# Start with delivered orders
master = orders_delivered.copy()

# Merge customers
master = master.merge(customers, on='customer_id', how='left')

# Merge order items
master = master.merge(order_items, on='order_id', how='left')

# Merge payments
payments_agg = payments.groupby('order_id').agg(
    total_payment  = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type   = ('payment_type', 'first')
).reset_index()
master = master.merge(payments_agg, on='order_id', how='left')

# Merge reviews
reviews_clean = reviews[['order_id', 'review_score']].copy()
master = master.merge(reviews_clean, on='order_id', how='left')

# Merge products (with English names)
products_clean = products[['product_id',
                            'product_category_name_english',
                            'product_weight_g']].copy()
master = master.merge(products_clean, on='product_id', how='left')

# Merge sellers
master = master.merge(sellers[['seller_id',
                                'seller_city',
                                'seller_state']], on='seller_id', how='left')

print(f"✅ Master dataset built!")
print(f"   Shape: {master.shape}")
print(f"   Columns: {list(master.columns)}")

✅ Master dataset built!
   Shape: (110189, 29)
   Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'is_late', 'days_diff', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'total_payment', 'payment_installments', 'payment_type', 'review_score', 'product_category_name_english', 'product_weight_g', 'seller_city', 'seller_state']


In [8]:
# ============================================================
# CELL 8 — SAVE CLEANED DATA
# ============================================================

# Save master dataset
master.to_csv('master_dataset.csv', index=False)

# Save cleaned individual tables
orders_delivered.to_csv('orders_clean.csv', index=False)
products.to_csv('products_clean.csv', index=False)
reviews.to_csv('reviews_clean.csv', index=False)

# Download all clean files to your Mac
files.download('master_dataset.csv')
files.download('orders_clean.csv')
files.download('products_clean.csv')
files.download('reviews_clean.csv')

print("✅ All clean files saved & downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All clean files saved & downloaded!


In [9]:
# ============================================================
# CELL 9 — PHASE 2 SUMMARY
# ============================================================

print("=" * 55)
print("        ✅ PHASE 2 CLEANING SUMMARY")
print("=" * 55)
print(f"  Master Dataset Rows    : {len(master):>10,}")
print(f"  Master Dataset Columns : {master.shape[1]:>10,}")
print(f"  Delivered Orders       : {len(orders_delivered):>10,}")
print(f"  Avg Delivery Days      : {master['delivery_days'].mean():>10.1f}")
print(f"  Late Delivery Rate     : {master['is_late'].mean()*100:>9.1f}%")
print(f"  Avg Review Score       : {master['review_score'].mean():>10.2f} / 5")
print(f"  Avg Order Value        : R${master['total_payment'].mean():>9.2f}")
print("=" * 55)
print("✅ Phase 2 Complete — Ready for SQL Analysis!")

        ✅ PHASE 2 CLEANING SUMMARY
  Master Dataset Rows    :    110,189
  Master Dataset Columns :         29
  Delivered Orders       :     96,470
  Avg Delivery Days      :       12.0
  Late Delivery Rate     :       7.9%
  Avg Review Score       :       4.08 / 5
  Avg Order Value        : R$   179.47
✅ Phase 2 Complete — Ready for SQL Analysis!
